# 🌲 Modelos de Ensamble y Redes Neuronales Multicapa (MLP)

### Autor: Néstor León | Business Intelligence & Analytics
---

Cuando la relación entre las variables de negocio y el objetivo es altamente no lineal o existen interacciones complejas, los modelos lineales tocan techo. Aquí entran en juego los **métodos de ensamble** (la combinación de múltiples árboles de decisión) y el **Deep Learning básico**.

En este notebook:
1. Comparamos **Random Forest (Bagging)** frente a **Gradient Boosting (Boosting secuencial)**.
2. Analizamos la **importancia de características** mediante *Permutation Importance* para no sesgarnos por la cardinalidad de los árboles.
3. Construimos un **Perceptrón Multicapa (MLP)** para clasificación tabular con control de sobreajuste mediante *Early Stopping*.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, accuracy_score

print("Módulos de Ensemble y Deep Learning importados.")


## 1. Simulación de un Problema de Negocio No Lineal

Construimos un caso de propensión donde existen fuertes interacciones cruzadas (ej. clientes con saldo alto pero poca antigüedad se comportan de forma distinta que con mucha antigüedad).


In [ ]:
np.random.seed(42)
N = 1500

X_df = pd.DataFrame({
    'antiguedad_meses': np.random.randint(1, 120, N),
    'saldo_medio': np.random.exponential(5000, N),
    'num_transacciones_mes': np.random.poisson(15, N),
    'reclamaciones_previas': np.random.choice([0, 1, 2, 3], N, p=[0.7, 0.2, 0.07, 0.03]),
    'uso_app_digital': np.random.choice([0, 1], N, p=[0.35, 0.65])
})

# Relación no lineal: Interacción entre antigüedad baja y alta reclamación incrementa drásticamente el target
score = (
    0.02 * X_df['antiguedad_meses'] +
    0.0003 * X_df['saldo_medio'] -
    1.2 * X_df['reclamaciones_previas'] +
    0.8 * X_df['uso_app_digital'] +
    0.0001 * (X_df['antiguedad_meses'] * X_df['num_transacciones_mes'])
)
prob = 1 / (1 + np.exp(-score + 2.0))
y = (prob > np.random.uniform(0, 1, N)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.25, random_state=42)
print(f"Dimensiones Train: {X_train.shape} | Tasa positiva: {y.mean()*100:.1f}%")


## 2. Entrenamiento Comparativo: Random Forest vs Gradient Boosting vs MLP

Ajustamos los tres paradigmas de modelado avanzado.


In [ ]:
# 1. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# 2. Gradient Boosting
gb = HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=42)
gb.fit(X_train, y_train)

# 3. Perceptrón Multicapa (MLP) con regularización L2 y Early Stopping
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=300, early_stopping=True, random_state=42)
mlp.fit(X_train, y_train)

modelos_adv = {'Random Forest': rf, 'Gradient Boosting': gb, 'Red Neuronal (MLP)': mlp}

for nombre, mod in modelos_adv.items():
    auc = roc_auc_score(y_test, mod.predict_proba(X_test)[:, 1])
    acc = accuracy_score(y_test, mod.predict(X_test))
    print(f"-> {nombre:20s} | ROC-AUC: {auc:.4f} | Accuracy: {acc*100:.2f}%")


## 3. Interpretabilidad mediante Permutation Importance

En entornos regulados no podemos entregar un modelo de caja negra sin explicar qué palancas mueven la predicción. La **importancia por permutación** calcula la caída en rendimiento al desordenar aleatoriamente los valores de cada columna.


In [ ]:
perm_importance = permutation_importance(gb, X_test, y_test, n_repeats=10, random_state=42)

df_imp = pd.DataFrame({
    'Caracteristica': X_df.columns,
    'Importancia': perm_importance.importances_mean
}).sort_values(by='Importancia', ascending=True)

plt.figure(figsize=(9, 4))
plt.barh(df_imp['Caracteristica'], df_imp['Importancia'], color='#16a085')
plt.title("Importancia de Variables (Permutation Importance - Gradient Boosting)", fontsize=13, fontweight='bold')
plt.xlabel("Caída Media en Precisión al Desordenar la Variable")
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()


## 4. Conclusiones Técnicas y Recomendaciones

1. **Gradient Boosting como estándar de facto**: En datos tabulares estructurados de negocio, Gradient Boosting suele superar a redes neuronales densas sin requerir una calibración hiperparamétrica tan costosa.
2. **Explicabilidad Garantizada**: Gracias a la importancia por permutación, podemos justificar ante la dirección que las *reclamaciones previas* y el *saldo medio* son los mayores condicionantes del comportamiento del usuario.
